In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yogape/logistics-operations-database")



In [3]:
database_schema_path = r"C:\Users\jinfe\.cache\kagglehub\datasets\yogape\logistics-operations-database\versions\1\DATABASE_SCHEMA.txt"
def show_schema():
    with open(database_schema_path, "r", encoding="utf-8") as file:
        print(file.read())

In [4]:
def load_raw_tables(base_path):
    customers = pd.read_csv(fr"{base_path}\customers.csv")
    delivery_events = pd.read_csv(fr"{base_path}\delivery_events.csv")
    drivers = pd.read_csv(fr"{base_path}\drivers.csv")
    loads = pd.read_csv(fr"{base_path}\loads.csv")
    routes = pd.read_csv(fr"{base_path}\routes.csv")
    trips = pd.read_csv(fr"{base_path}\trips.csv")
    trucks = pd.read_csv(fr"{base_path}\trucks.csv")
    facilities = pd.read_csv(fr"{base_path}\facilities.csv")

    return {
        "customers": customers,
        "delivery_events": delivery_events,
        "drivers": drivers,
        "loads": loads,
        "routes": routes,
        "trips": trips,
        "trucks": trucks,
        "facilities": facilities,
    }


In [5]:
def prepare_delivery_events(delivery_events):
    delivery_events = delivery_events.copy()

    delivery_events["actual_datetime"] = pd.to_datetime(
        delivery_events["actual_datetime"],
        errors="coerce"
    )
    delivery_events["hour_of_day"] = delivery_events["actual_datetime"].dt.hour
    delivery_events["day_of_week"] = delivery_events["actual_datetime"].dt.day_name()

    congestion = (
        delivery_events
        .groupby(["facility_id", "day_of_week", "hour_of_day"])
        .size()
        .reset_index(name="facility_congestion_score")
    )

    detention = (
        delivery_events
        .groupby(["facility_id", "day_of_week", "hour_of_day"])["detention_minutes"]
        .mean()
        .reset_index(name="facility_detention_avg")
    )

    delivery_events = delivery_events.merge(
        congestion,
        on=["facility_id", "day_of_week", "hour_of_day"],
        how="left"
    )

    delivery_events = delivery_events.merge(
        detention,
        on=["facility_id", "day_of_week", "hour_of_day"],
        how="left"
    )

    return delivery_events

In [6]:
def build_master_df(delivery_events, trips, loads, drivers, trucks, routes, customers):
    df = delivery_events.copy()

    df = df.merge(trips.drop(columns=["load_id"]), on="trip_id", how="left")
    df = df.merge(loads, on="load_id", how="left")
    df = df.merge(
        drivers[["driver_id", "years_experience", "home_terminal"]],
        on="driver_id",
        how="left"
    )
    df = df.merge(
        trucks[["truck_id", "make", "model_year"]],
        on="truck_id",
        how="left"
    )
    df = df.merge(
        routes[["route_id", "typical_distance_miles", "typical_transit_days"]],
        on="route_id",
        how="left"
    )
    df = df.merge(
        customers[["customer_id", "customer_type"]],
        on="customer_id",
        how="left"
    )

    return df

In [7]:
def add_engineering_features(df):
    df = df.copy()
    df['month'] = df['actual_datetime'].dt.month
    df["truck_age_at_event"] = df["actual_datetime"].dt.year - df["model_year"]
    df["distance_deviation"] = df["actual_distance_miles"] - df["typical_distance_miles"]
    df["is_home_terminal"] = (df["home_terminal"] == df["location_city"]).astype(int)

    df["detention_risk_index"] = df["facility_congestion_score"] * df["facility_detention_avg"]
    df["efficiency_ratio"] = df["average_mpg"] / (df["truck_age_at_event"] + 1)
    df["revenue_density"] = df["revenue"] / (df["weight_lbs"] + 1)
    df['is_winter'] = df['month'].apply(lambda x: 1 if x in [12, 1, 2] else 0)
    
    return df